# Reconstruct pool state block-by-block

Turn the irregular swap events into one aligned, gap-free per-pool series
(price + active liquidity), trim to the study window, and save one CSV per pool
under `S.processed_dir`. Parameters come from `arblib.config.STUDY`.

In [1]:
%load_ext autoreload
%autoreload 2

from arblib import data_io, preprocessing as pp
from arblib.config import STUDY as S, LIQUIDITY_FILES

## 0. Load the raw swap extracts

In [2]:
dfs = data_io.load_pool_csvs(S.swaps_dir)

Loaded: df_uniswap_swap.csv
       amount0               amount1         dex  evt_block_number  \
0   -999266400    334197952309227723  uniswap_v3          24133363   
1     33925490    -11345521500000000  uniswap_v4          24133364   
2  13936469654  -4658615254546375464  uniswap_v3          24133365   
3   -107703773     36025562985474874  uniswap_v3          24133365   
4   -154244910     51582178535019641  uniswap_v4          24133365   

                evt_block_time  evt_index  \
0  2025-12-31 15:00:11.000 UTC         17   
1  2025-12-31 15:00:23.000 UTC          3   
2  2025-12-31 15:00:35.000 UTC        353   
3  2025-12-31 15:00:35.000 UTC         40   
4  2025-12-31 15:00:35.000 UTC        719   

                                         evt_tx_hash  fee  \
0  0x045ccf214364711f41a1e4ebdc460703fb1d3f6ccfae...  100   
1  0x7e40441f67a6ffeb29115fdf3028481faf2a2b9772bf...    0   
2  0xa8e2974554c1b06a7ab79cf43f0764f29ea521b0e1e6...  500   
3  0xa0671b25449576f08044498e74206b1

## 1. Keep the end-of-block price per pool & block

In [3]:
dfs = pp.count_swaps(dfs)
filtered_dfs = pp.clean_all(dfs)

Counted swaps df_uniswap: 1066 rows
Counted swaps df_pancake: 240 rows
Processed df_uniswap: 1066 rows -> 605 rows
Processed df_pancake: 240 rows -> 180 rows


## 2. Split each DEX into one series per pool

In [4]:
pool_dfs = pp.split_by_pool(filtered_dfs)

Created uniswap_1: 75 rows
Created uniswap_2: 128 rows
Created uniswap_3: 24 rows
Created uniswap_4: 3 rows
Created uniswap_5: 230 rows
Created uniswap_6: 145 rows
Created pancake_1: 137 rows
Created pancake_2: 43 rows

Total: 8 dataframes
['uniswap_1', 'uniswap_2', 'uniswap_3', 'uniswap_4', 'uniswap_5', 'uniswap_6', 'pancake_1', 'pancake_2']


## 3. Drop pools that trade too rarely to reconstruct

In [5]:
pool_dfs, dropped_pools = pp.filter_pools_by_swap_gap(pool_dfs, S.max_gap_blocks)

k = 6000 blocks
Kept 8 pools

Kept pools:
  uniswap_1: 75 swaps, max consecutive gap = 16 blocks
  uniswap_2: 128 swaps, max consecutive gap = 11 blocks
  uniswap_3: 24 swaps, max consecutive gap = 49 blocks
  uniswap_4: 3 swaps, max consecutive gap = 107 blocks
  uniswap_5: 230 swaps, max consecutive gap = 5 blocks
  uniswap_6: 145 swaps, max consecutive gap = 9 blocks
  pancake_1: 137 swaps, max consecutive gap = 9 blocks
  pancake_2: 43 swaps, max consecutive gap = 24 blocks


## 4. Reconstruct a dense, forward-filled series per pool

In [6]:
reconstructed_pools, global_min, global_max, block_time_map = pp.reconstruct_pool_timeseries(pool_dfs)

Global block range: 24133363 to 24133662
Total blocks: 300

uniswap_1: 75 trades -> 295 blocks (98.3%)
uniswap_2: 128 trades -> 298 blocks (99.3%)
uniswap_3: 24 trades -> 297 blocks (99.0%)
uniswap_4: 3 trades -> 235 blocks (78.3%)
uniswap_5: 230 trades -> 300 blocks (100.0%)
uniswap_6: 145 trades -> 299 blocks (99.7%)
pancake_1: 137 trades -> 298 blocks (99.3%)
pancake_2: 43 trades -> 290 blocks (96.7%)

Created 8 reconstructed time series


## 4b. Reconstruct active liquidity per block

Rebuild each pool's `liquidity` into the running active-liquidity state using the
mint/burn events (in-range deltas applied between swaps, held constant otherwise).

In [7]:
liq_dfs = data_io.load_pool_csvs(S.liquidity_dir, files=LIQUIDITY_FILES)
reconstructed_pools = pp.reconstruct_liquidity_states(reconstructed_pools, liq_dfs)

Loaded: df_uniswap_liq.csv
          dex  evt_block_number               evt_block_time  evt_index  \
0  uniswap_v3          24133387  2025-12-31 15:04:59.000 UTC        557   
1  uniswap_v3          24133406  2025-12-31 15:08:47.000 UTC        179   
2  uniswap_v3          24133460  2025-12-31 15:19:35.000 UTC        318   
3  uniswap_v3          24133591  2025-12-31 15:45:47.000 UTC        141   
4  uniswap_v3          24133626  2025-12-31 15:52:47.000 UTC        198   

        liquidity_delta                                        pool  \
0        -1291771469796  0xe0554a476a092703abdb3ef35c80e0d76d32939f   
1      5244061520071531  0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640   
2  28535564165621841753  0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640   
3       -49393047801584  0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640   
4      -509787476461393  0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8   

   tick_lower  tick_upper  
0      194973      197604  
1      196250      196730  
2      1963

## 5. Trim to the study window

In [8]:
filtered_pools = pp.filter_by_start_time(reconstructed_pools, S.study_start)

uniswap_1: 300 -> 225 rows
uniswap_2: 300 -> 225 rows
uniswap_3: 300 -> 225 rows
uniswap_4: 300 -> 225 rows
uniswap_5: 300 -> 225 rows
uniswap_6: 300 -> 225 rows
pancake_1: 300 -> 225 rows
pancake_2: 300 -> 225 rows

Filtered all pools by time >= 2025-12-31 15:15:00


## 6. Save one CSV per pool

In [9]:
data_io.save_processed_pools(filtered_pools, S.processed_dir)

Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/processed/uniswap_1.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/processed/uniswap_2.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/processed/uniswap_3.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/processed/uniswap_4.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/processed/uniswap_5.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/processed/uniswap_6.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/processed/pancake_1.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/processed/pancake_2.csv
Done.
